
# MOovimiento Circular......


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Origen de la Aceleración Centrípeta</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Utiliza los controles inferiores para mostrar u ocultar los vectores. Observa cómo, al hacer <strong>Δt → 0</strong>, la aceleración media encaja exactamente en la aceleración instantánea (hacia el centro).
    </p>
    
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir Giro</button>
            <button id="limit_btn_UID" style="background-color: #6f42c1; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">🎯 Animar Límite (Δt → 0)</button>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc;">
                <label style="font-weight: 600; font-size: 0.85em; margin-bottom: 5px; color: #dc3545;">Intervalo de Tiempo (Δt)</label>
                <input type="range" id="dt_slider_UID" min="0.001" max="1.5" step="0.01" value="1.0" style="width: 150px;">
                <span id="dt_val_UID" style="font-weight: bold; font-family: monospace; font-size: 0.9em;">1.000 s</span>
            </div>
        </div>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 15px;">
        <canvas id="canvas_sim_UID" width="550" height="450" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="font-size: 0.95em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v1_UID" checked style="cursor: pointer;">
            <span><b style="color:#0d6efd;">▬ v₁</b> (Vel. actual)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v2_UID" checked style="cursor: pointer;">
            <span><b style="color:#0dcaf0;">▬ v₂</b> (Vel. futura)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_dv_UID" style="cursor: pointer;">
            <span><b style="color:#fd7e14;">▬ Δv</b> (Resta de vectores)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_am_UID" style="cursor: pointer;">
            <span><b style="color:#6f42c1;">▬ a_media</b> (Δv/Δt)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_ac_UID" style="cursor: pointer;">
            <span><b style="color:#dc3545;">▬ a_instantánea</b> (Real)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_limit = document.getElementById('limit_btn_UID');
            const dt_slider = document.getElementById('dt_slider_UID');
            const dt_val = document.getElementById('dt_val_UID');
            
            // Elementos de visibilidad (Checkboxes)
            const chk_v1 = document.getElementById('chk_v1_UID');
            const chk_v2 = document.getElementById('chk_v2_UID');
            const chk_dv = document.getElementById('chk_dv_UID');
            const chk_am = document.getElementById('chk_am_UID');
            const chk_ac = document.getElementById('chk_ac_UID');
            
            let isPlaying = false;
            let limitAnimationActive = false;
            let animationId;
            let lastTime = 0; 
            
            // Físicas Base Fijas 
            const r = 2.0;       // Radio
            const omega = 1.0;   // Velocidad angular (1 rad/s)
            const v = omega * r;
            const ac = omega * omega * r;
            
            let t = 0;           // Tiempo actual
            let dt = parseFloat(dt_slider.value); // Intervalo delta t
            
            // Función para dibujar flechas
            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const headlen = 12; // Aumentado ligeramente para balancear los vectores más grandes
                const angle = Math.atan2(toy - fromy, tox - fromx);
                
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                if(dashed) context.setLineDash([5, 5]);
                else context.setLineDash([]);
                
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                context.setLineDash([]); 
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function drawSim() {
                // Cálculos Físicos
                const t2 = t + dt;
                const theta1 = omega * t;
                const theta2 = omega * t2;
                
                // Posiciones reales
                const x1 = r * Math.cos(theta1),  y1 = r * Math.sin(theta1);
                const x2 = r * Math.cos(theta2),  y2 = r * Math.sin(theta2);
                
                // Velocidades 
                const v1x = -v * Math.sin(theta1), v1y = v * Math.cos(theta1);
                const v2x = -v * Math.sin(theta2), v2y = v * Math.cos(theta2);
                
                // Variación de velocidad y aceleración media
                const dvx = v2x - v1x, dvy = v2y - v1y;
                const amx = dvx / dt,  amy = dvy / dt;
                
                // Aceleración instantánea exacta
                const acx = -ac * Math.cos(theta1), acy = -ac * Math.sin(theta1);
                
                // Renderizado Canvas
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                const cx = canvas.width / 2;
                const cy = canvas.height / 2;
                
                // Factores de escala visuales (¡Aumentados para vectores mucho más grandes!)
                const Sp = 140 / r;           // Escala posición de la órbita
                const Sv = 110 / v;           // Escala velocidad (antes 50)
                const Sa = 120 / ac;          // Escala aceleración (antes 60)

                // Dibujar Ejes de Referencia
                ctx.strokeStyle = "#eee"; ctx.lineWidth = 1;
                ctx.beginPath(); ctx.moveTo(0, cy); ctx.lineTo(canvas.width, cy);
                ctx.moveTo(cx, 0); ctx.lineTo(cx, canvas.height); ctx.stroke();

                // Trayectoria Circular
                ctx.beginPath(); ctx.strokeStyle = "#aaa"; ctx.setLineDash([4, 4]);
                ctx.arc(cx, cy, r * Sp, 0, 2 * Math.PI); ctx.stroke(); ctx.setLineDash([]);

                // Coordenadas transformadas al canvas (Eje Y invertido)
                const c_x1 = cx + x1 * Sp, c_y1 = cy - y1 * Sp;
                const c_x2 = cx + x2 * Sp, c_y2 = cy - y2 * Sp;

                // Coordenadas de las puntas de los vectores
                const p_v1_x = c_x1 + v1x * Sv, p_v1_y = c_y1 - v1y * Sv;
                const p_v2_x = c_x2 + v2x * Sv, p_v2_y = c_y2 - v2y * Sv;
                const p_v2_trans_x = c_x1 + v2x * Sv, p_v2_trans_y = c_y1 - v2y * Sv; // v2 trasladado a pos 1

                // 1. Dibujar Aceleración Instantánea
                if (chk_ac.checked) {
                    drawArrow(ctx, c_x1, c_y1, c_x1 + acx * Sa, c_y1 - acy * Sa, "#dc3545", 3);
                }

                // 2. Dibujar Aceleración Media
                if (chk_am.checked) {
                    drawArrow(ctx, c_x1, c_y1, c_x1 + amx * Sa, c_y1 - amy * Sa, "#6f42c1", 3);
                }

                if (dt > 0.01) {
                    // Dibujar v2 en la posición 2
                    if (chk_v2.checked) {
                        drawArrow(ctx, c_x2, c_y2, p_v2_x, p_v2_y, "#0dcaf0", 3);
                        // Bolita 2 (Sombra)
                        ctx.beginPath(); ctx.fillStyle = "rgba(0,0,0,0.2)";
                        ctx.arc(c_x2, c_y2, 7, 0, Math.PI * 2); ctx.fill();
                    }
                    
                    // Dibujar Traslación de v2 y Resta Vectorial (Δv)
                    if (chk_dv.checked) {
                        drawArrow(ctx, c_x1, c_y1, p_v2_trans_x, p_v2_trans_y, "#0dcaf0", 3, true);
                        drawArrow(ctx, p_v1_x, p_v1_y, p_v2_trans_x, p_v2_trans_y, "#fd7e14", 3);
                    }
                }

                // 3. Dibujar v1 en la posición 1
                if (chk_v1.checked) {
                    drawArrow(ctx, c_x1, c_y1, p_v1_x, p_v1_y, "#0d6efd", 3);
                }

                // Bolita 1 (Posición actual, siempre visible)
                ctx.beginPath(); ctx.fillStyle = "#333";
                ctx.arc(c_x1, c_y1, 10, 0, Math.PI * 2); ctx.fill();
            }

            // Actualizar canvas cuando cambia un checkbox (si no está animándose)
            [chk_v1, chk_v2, chk_dv, chk_am, chk_ac].forEach(chk => {
                chk.addEventListener('change', () => {
                    if (!isPlaying && !limitAnimationActive) drawSim();
                });
            });

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                if (isPlaying) {
                    t += deltaTime * 0.5; // Velocidad de rotación
                }
                
                if (limitAnimationActive) {
                    dt = dt * 0.99; // Límite suave
                    if (dt <= 0.005) {
                        dt = 0.001; 
                        limitAnimationActive = false;
                        btn_limit.innerText = "🎯 Animar Límite (Δt → 0)";
                        btn_limit.style.backgroundColor = "#6f42c1";
                    }
                    dt_slider.value = dt;
                    dt_val.innerText = dt.toFixed(3) + " s";
                }
                
                drawSim();
                
                if(isPlaying || limitAnimationActive) {
                    animationId = requestAnimationFrame(animate);
                }
            }

            dt_slider.addEventListener('input', () => {
                dt = parseFloat(dt_slider.value);
                dt_val.innerText = dt.toFixed(3) + " s";
                limitAnimationActive = false;
                btn_limit.innerText = "🎯 Animar Límite (Δt → 0)";
                btn_limit.style.backgroundColor = "#6f42c1";
                if(!isPlaying && !limitAnimationActive) drawSim();
            });

            btn_play.addEventListener('click', () => {
                isPlaying = !isPlaying;
                if (isPlaying) {
                    btn_play.innerText = "⏸ Pausar Giro";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                    lastTime = 0; 
                    animationId = requestAnimationFrame(animate);
                } else {
                    btn_play.innerText = "▶ Reproducir Giro";
                    btn_play.style.backgroundColor = "#198754";
                    btn_play.style.color = "#fff";
                }
            });

            btn_limit.addEventListener('click', () => {
                if(dt <= 0.01) {
                    dt = 1.0;
                    dt_slider.value = dt;
                }
                limitAnimationActive = true;
                btn_limit.innerText = "⏳ Acercando...";
                btn_limit.style.backgroundColor = "#6c757d";
                lastTime = 0;
                
                if (!isPlaying) animationId = requestAnimationFrame(animate);
            });
            
            drawSim();
        }
        
        initSimulation();
    })();
</script>
"""

# Reemplazamos el identificador para evitar conflictos y mostramos
html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))